# NeuroTrain Lab — Notebook 4: Entrenamiento y Sobreajuste

**Tema:** cómo organizar un entrenamiento real de principio a fin —
split train/validation/test, escalado sin fuga de información, un
baseline honesto, `Dropout` y `EarlyStopping` — y cómo detectar el
sobreajuste leyendo las curvas de aprendizaje.

> Último notebook de 4. Ya conoces el perceptrón (NB1), la pérdida y
> el backprop (NB2), y los optimizadores (NB3). Aquí juntamos todo
> para entrenar un modelo real sobre un problema real y aprender a
> **controlar** su sobreajuste, no solo observarlo.

## 🎯 Qué aprenderás en este notebook

Al terminar podrás explicar, sin fórmulas de memoria:

1. Por qué se separan tres conjuntos (train/validation/test) y no dos.
2. Por qué el escalado se ajusta solo con train.
3. Qué hacen `Dropout` y `EarlyStopping`, y por qué se combinan.
4. Cómo leer `loss` y `val_loss` para diagnosticar sobreajuste.
5. Por qué siempre comparamos contra un baseline más simple.

**Mapa mental:** `datos reales → split → escalado → baseline → MLP → EarlyStopping+Dropout → fit() → curvas → test vs baseline → experimento A/B`

In [ ]:
from pathlib import Path
import json
import math
import sys

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "breast_cancer_wisconsin.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from neurotrain.celebrations import celebrate
from neurotrain.config import TrainingConfig
from neurotrain.data import load_dataset, prepare_data
from neurotrain.evaluation import classification_metrics
from neurotrain.modeling import train_dense_classifier
from neurotrain.visualization import (
    plot_confusion,
    plot_roc,
    plot_training_history,
    plot_training_history_comparison,
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.keras.utils.set_random_seed(RANDOM_STATE)

print("TensorFlow:", tf.__version__)
print("Raíz del proyecto:", PROJECT_ROOT)

## 1. Auditoría rápida del dataset

Ya viste este CSV de pasada en el Notebook 1. Antes de entrenar
confirmamos su contrato mínimo con `assert`: forma, ausencia de
nulos y las dos etiquetas esperadas. Es la última vez que lo hacemos
"a mano" — el resto del proyecto usa `neurotrain.data.load_dataset()`,
que hace exactamente estas comprobaciones.

In [ ]:
DATA_PATH = PROJECT_ROOT / "data" / "breast_cancer_wisconsin.csv"
df = pd.read_csv(DATA_PATH)

assert df.shape == (569, 31)
assert not df.isna().any().any()
assert set(df["diagnosis"].unique()) == {"B", "M"}

class_counts = df["diagnosis"].value_counts().rename(index={"B": "Benigno", "M": "Maligno"})
print(f"Filas: {df.shape[0]} | Columnas: {df.shape[1]}")
display(class_counts.to_frame("registros"))

## 2. Separar X e y

La clase positiva se define explícitamente como **maligno = 1** —
así "sensibilidad" significa "proporción de malignos reales que
detectamos", sin ambigüedad.

In [ ]:
X = df.drop(columns="diagnosis")
y = df["diagnosis"].eq("M").astype("int8")

print("Forma de X:", X.shape, "| Forma de y:", y.shape)

## 3. Crear train, validation y test

<div style="border-left:4px solid #7C3AED; background:#F5F3FF; border-radius:.4rem; padding:.85rem 1.1rem; margin:.7rem 0;">
<b>🧠 CONCEPTO CLAVE — ¿Por qué tres conjuntos y no dos?</b><br><br>
`train` son los ejercicios con los que la red ajusta sus pesos.
`validation` son simulacros: la red no aprende de ellos directamente,
pero **nosotros** los usamos para decidir arquitectura, dropout,
paciencia o umbral. `test` es el examen final que solo se abre
**una vez**, al terminar. Si usas test para decidir nada, ya no mide
lo que dice medir: solo mide qué tan bien te ajustaste al examen.
</div>

Usaremos aproximadamente 70% / 15% / 15%. `stratify=y` mantiene una
proporción parecida de benignos y malignos en cada partición.

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE,
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_STATE,
)

split_summary = pd.DataFrame(
    {
        "registros": [len(X_train), len(X_val), len(X_test)],
        "% malignos": [y_train.mean(), y_val.mean(), y_test.mean()],
    },
    index=["train", "validation", "test"],
)
display(split_summary.style.format({"% malignos": "{:.1%}"}))

## 4. Escalar sin fuga de información

<div style="border-left:4px solid #F97316; background:#FFF7ED; border-radius:.4rem; padding:.85rem 1.1rem; margin:.7rem 0;">
<b>⚠️ ERROR TÍPICO — El error más común: escalar antes de dividir</b><br><br>
Si ajustas `StandardScaler` con **todo** el dataset y luego divides,
la media y desviación de train ya "vieron" ejemplos de validation y
test. Es una fuga de información sutil: el modelo no copia
respuestas, pero su preprocesado sí se benefició del examen final.
La regla es siempre: `fit_transform` solo en train, `transform` en
el resto.
</div>

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).astype("float32")
X_val_scaled = scaler.transform(X_val).astype("float32")
X_test_scaled = scaler.transform(X_test).astype("float32")

print("Media aproximada de train:", X_train_scaled.mean(axis=0)[:3].round(5))
print("Forma que recibirá la red:", X_train_scaled.shape)

<div style="text-align:center; opacity:.85; font-style:italic; margin:1.1rem 0; font-size:1.05rem;">🌱 Acabas de sembrar la primera semilla de tu red neuronal.</div>

## 5. Crear un baseline honesto

Una regresión logística responde a la misma pregunta y es mucho más
simple. Si rinde igual o mejor que la red, la conclusión profesional
no es "la ANN falló": es "la complejidad adicional no se justificó
con estos datos".

In [ ]:
baseline = LogisticRegression(max_iter=2_000, random_state=RANDOM_STATE)
baseline.fit(X_train_scaled, y_train)

baseline_probabilities = baseline.predict_proba(X_test_scaled)[:, 1]
print("ROC-AUC baseline:", round(roc_auc_score(y_test, baseline_probabilities), 3))

## 6. Epochs, batches e iteraciones

Ya usaste el vocabulario "batch"/"step" en el Notebook 3 al hablar de
optimizadores. Aquí solo lo aterrizamos en números concretos para
**este** split: con `X_train_scaled` de tamaño fijo y un
`batch_size` dado, ¿cuántas actualizaciones de pesos ocurren por
época?

In [ ]:
BATCH_SIZE = 32
EPOCHS = 200

### ✏️ Ejercicio

Completa el cálculo de actualizaciones por época **usando el tamaño
real de `X_train_scaled`**, no un número fijo. Recuerda: una
actualización ocurre por cada batch procesado, y el último batch de
la época puede ser más pequeño (por eso se redondea hacia arriba
con `math.ceil`).

In [ ]:
updates_per_epoch = math.ceil(✏️✏️✏️)
max_updates = updates_per_epoch * EPOCHS

print("Actualizaciones por época:", updates_per_epoch)
print("Actualizaciones máximas (si se completan todas las épocas):", max_updates)

<details>
<summary><b>Ver solución</b></summary>

```python
updates_per_epoch = math.ceil(len(X_train_scaled) / BATCH_SIZE)
max_updates = updates_per_epoch * EPOCHS

print("Actualizaciones por época:", updates_per_epoch)
print("Actualizaciones máximas (si se completan todas las épocas):", max_updates)
```

</details>

## 7. Construir la red

`30 variables → Dense(32, ReLU) → Dropout(0.30) → Dense(16, ReLU) → Dense(1, Sigmoid)`

El porqué de 32 y 16 neuronas (hiperparámetros que se validan, no
fórmulas del número de variables) ya lo viste en el Notebook 1. Lo
único nuevo aquí es la capa de salida: una neurona con activación
**Sigmoid** produce la probabilidad que usamos como predicción.

### ✏️ Ejercicio

Completa la activación de la capa de salida. Pista: necesitamos un
único número entre 0 y 1 interpretable como probabilidad de
"maligno" — la misma función que usaste en el Notebook 1 para la
capa de salida binaria.

In [ ]:
model = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=(X_train_scaled.shape[1],)),
        tf.keras.layers.Dense(32, activation="relu"),
        tf.keras.layers.Dropout(0.30),
        tf.keras.layers.Dense(16, activation="relu"),
        tf.keras.layers.Dense(1, activation="✏️✏️✏️"),
    ],
    name="neurotrain_mlp",
)
model.summary()

<details>
<summary><b>Ver solución</b></summary>

```python
model = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=(X_train_scaled.shape[1],)),
        tf.keras.layers.Dense(32, activation="relu"),
        tf.keras.layers.Dropout(0.30),
        tf.keras.layers.Dense(16, activation="relu"),
        tf.keras.layers.Dense(1, activation="sigmoid"),
    ],
    name="neurotrain_mlp",
)
model.summary()
```

</details>

## 8. Compilar: optimizer, loss y métricas

`compile()` todavía no entrena, solo configura las reglas. El
optimizer (Adam) lo estudiaste a fondo en el Notebook 3; la loss
(binary cross-entropy) la estudiaste a fondo en el Notebook 2. Aquí
solo los conectamos con métricas que sí observamos pero que no
sustituyen a la loss.

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.AUC(name="roc_auc"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="sensitivity"),
    ],
)

## 9. EarlyStopping y Dropout

<div style="border-left:4px solid #7C3AED; background:#F5F3FF; border-radius:.4rem; padding:.85rem 1.1rem; margin:.7rem 0;">
<b>🧠 CONCEPTO CLAVE — Dropout: forzar redundancia</b><br><br>
`Dropout(0.30)` apaga al azar el 30% de las neuronas de esa capa en
**cada paso de entrenamiento**. Obliga a la red a no depender
siempre de las mismas rutas internas — el equivalente a estudiar
sin memorizar el orden exacto de las preguntas. En inferencia
(predicción real) no se apaga ninguna neurona.
</div>

<div style="border-left:4px solid #7C3AED; background:#F5F3FF; border-radius:.4rem; padding:.85rem 1.1rem; margin:.7rem 0;">
<b>🧠 CONCEPTO CLAVE — EarlyStopping: dejar de entrenar en el momento justo</b><br><br>
Observa `val_loss` época a época. Si no mejora durante `patience`
épocas consecutivas, detiene el entrenamiento.
`restore_best_weights=True` recupera los pesos de la **mejor**
época observada, no los de la última — así un empeoramiento tardío
no se queda como resultado final.
</div>

<div style="border-left:4px solid #2563EB; background:#EFF6FF; border-radius:.4rem; padding:.85rem 1.1rem; margin:.7rem 0;">
<b>❓ DUDA PROBABLE — ¿Validation también entrena a la red?</b><br><br>
No. `validation_data` se evalúa al final de cada época solo para
**medir**; sus ejemplos nunca participan en el cálculo del
gradiente ni actualizan pesos. Por eso puede usarse para decidir
cuándo parar sin "hacer trampa".
</div>

In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=12,
    restore_best_weights=True,
    verbose=1,
)

## 10. Entrenar con `fit()`

En cada batch ocurren, en una línea, los cuatro pasos que ya
diseccionaste en los Notebooks 2 y 3: forward pass → loss →
backpropagation → paso del optimizer.

In [ ]:
history = model.fit(
    X_train_scaled,
    y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stopping],
    verbose=0,
)

print(f"Épocas ejecutadas: {len(history.history['loss'])} de {EPOCHS}")

## 11. Leer las curvas de aprendizaje

- Si `loss` y `val_loss` bajan juntas, la red aprende patrones que
  generalizan.
- Si `loss` sigue bajando mientras `val_loss` sube, la red está
  **memorizando** train: sobreajuste.
- Si ambas quedan altas, puede haber infraajuste, pocas épocas o
  una configuración inadecuada.

No mires solo `accuracy`: con clases desbalanceadas puede parecer
buena aunque el modelo falle justo en la clase que importa.

In [ ]:
history_dict = {key: list(values) for key, values in history.history.items()}
fig = plot_training_history(history_dict, lang="es")
plt.show()

## 12. Evaluar una sola vez en test, frente al baseline

Umbral inicial 0.50: probabilidad ≥ 0.50 se convierte en "maligno"
(1). Sensibilidad, especificidad, precisión y ROC-AUC ya las
calculó `classification_metrics` — la misma lógica que escribirías
a mano, empaquetada para no repetirla en cada notebook.

In [ ]:
THRESHOLD = 0.50
probabilities = model.predict(X_test_scaled, verbose=0).ravel()

ann_metrics = classification_metrics(y_test, probabilities, THRESHOLD)
baseline_metrics = classification_metrics(y_test, baseline_probabilities, THRESHOLD)

comparison = pd.DataFrame(
    {"Red (MLP)": ann_metrics, "Baseline (LogReg)": baseline_metrics}
).loc[["accuracy", "roc_auc", "precision", "sensitivity", "specificity", "f1"]]
display(comparison.style.format("{:.3f}"))

In [ ]:
fig_confusion = plot_confusion(y_test, probabilities, threshold=THRESHOLD, lang="es")
plt.show()

fig_roc = plot_roc(y_test, probabilities, lang="es")
plt.show()

<div style="border-left:4px solid #22C55E; background:#F0FDF4; border-radius:.4rem; padding:.85rem 1.1rem; margin:.7rem 0;">
<b>📌 PARA RECORDAR — Ganar al baseline no es opcional para justificar la red</b><br><br>
Si la red no supera de forma clara a la regresión logística, la
decisión profesional correcta suele ser **usar el modelo más
simple**: es más barato de entrenar, más fácil de explicar y menos
propenso a sobreajustar con pocos datos.
</div>

## 13. Experimento guiado de sobreajuste (A vs B)

En vez de reeditar las celdas anteriores (arriesgando perder tu
referencia), vamos a lanzar **dos configuraciones independientes**
que coexisten, usando `TrainingConfig`:

| Variante | Capas | Dropout | EarlyStopping | Hipótesis |
|---|---:|---:|---:|---|
| A | 128 → 64 | 0.0 | No | Train mejorará; validation puede empeorar |
| B | 32 → 16 | 0.30 | Sí | Menos capacidad de memorizar, parada más temprana |

La variante B es, de hecho, la misma arquitectura que acabas de
entrenar a mano en las Secciones 7-10 — aquí la reproducimos vía
`TrainingConfig` para que quede lado a lado con A.

Antes de ejecutar, **escribe tu predicción** en una celda de texto
propia: ¿cuál crees que tendrá menor `val_loss` mínima?

`load_dataset()` y `prepare_data()` son la **misma lógica** que
escribiste a mano en las Secciones 1, 3 y 4 (auditoría, split
estratificado, `StandardScaler` ajustado solo con train) —
empaquetada para que un proyecto real no la repita en cada
experimento.

In [ ]:
frame = load_dataset()
data = prepare_data(frame, random_state=RANDOM_STATE)
print("Train:", data.X_train.shape, "| Val:", data.X_val.shape, "| Test:", data.X_test.shape)

### ✏️ Ejercicio

Completa `config_a` siguiendo la hipótesis de la tabla: sin
Dropout y sin EarlyStopping, para que la red pueda memorizar train
libremente durante las 200 épocas.

In [ ]:
config_a = TrainingConfig(
    hidden_units=(128, 64),
    dropout_rate=✏️✏️✏️,
    use_early_stopping=✏️✏️✏️,
    epochs=200,
    batch_size=32,
    random_state=RANDOM_STATE,
)
model_a, history_a = train_dense_classifier(data, config_a)
print("Épocas ejecutadas (A):", len(history_a["loss"]))

<details>
<summary><b>Ver solución</b></summary>

```python
config_a = TrainingConfig(
    hidden_units=(128, 64),
    dropout_rate=0.0,
    use_early_stopping=False,
    epochs=200,
    batch_size=32,
    random_state=RANDOM_STATE,
)
model_a, history_a = train_dense_classifier(data, config_a)
print("Épocas ejecutadas (A):", len(history_a["loss"]))
```

</details>

In [ ]:
config_b = TrainingConfig(
    hidden_units=(32, 16),
    dropout_rate=0.30,
    use_early_stopping=True,
    patience=12,
    epochs=200,
    batch_size=32,
    random_state=RANDOM_STATE,
)
model_b, history_b = train_dense_classifier(data, config_b)
print("Épocas ejecutadas (B):", len(history_b["loss"]))

<div style="text-align:center; opacity:.85; font-style:italic; margin:1.1rem 0; font-size:1.05rem;">🏁 Última recta antes del final del notebook.</div>

In [ ]:
fig_comparison = plot_training_history_comparison(
    history_a,
    history_b,
    "A: 128→64, sin regularización",
    "B: 32→16, con Dropout+EarlyStopping",
    lang="es",
)
plt.show()

Responde con la gráfica delante:

1. ¿En qué época fue mínima `val_loss` en cada variante?
2. ¿Cuánto se separaron `loss` y `val_loss` en A? ¿Y en B?
3. ¿La red más grande (A) mejoró el resultado en test, o solo en train?
4. ¿Alguna de las dos variantes justificó ser más compleja que el baseline de la Sección 5?

## 14. Guardar el modelo y el preprocesado

Un modelo sin su scaler no reproduce el mismo flujo: guardamos
ambos y metadatos mínimos del experimento de referencia (la
variante B, que fue la que evaluamos en test en la Sección 12).

In [ ]:
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
ARTIFACTS_DIR.mkdir(exist_ok=True)

model.save(ARTIFACTS_DIR / "neurotrain_model.keras")
joblib.dump(scaler, ARTIFACTS_DIR / "scaler.joblib")

metadata = {
    "dataset": "UCI Breast Cancer Wisconsin Diagnostic",
    "positive_class": "M = 1",
    "feature_names": X.columns.tolist(),
    "threshold": THRESHOLD,
    "epochs_executed": len(history.history["loss"]),
    "test_metrics": ann_metrics,
    "intended_use": "educational demonstration only",
}
(ARTIFACTS_DIR / "metadata.json").write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("Artefactos guardados en:", ARTIFACTS_DIR)

## 15. Del notebook al producto

Todo lo que hiciste en estos 4 notebooks vive también en una app
Streamlit con un **recorrido guiado**: una página "Inicio" y una
página por tema (Perceptrón, Pérdida y backprop, Optimizadores,
Entrenamiento — este mismo notebook, resumido de forma visual).

Además tiene un **"Modo Experimento"**: una página de laboratorio
donde puedes reentrenar de forma interactiva — cambiando
arquitectura, dropout, épocas, paciencia y umbral — con progreso
en vivo por época y un panel que revela el código real detrás de
cada botón.

```powershell
streamlit run app.py
```

**Siguiente paso:** abre la app, reproduce las variantes A y B
desde el Modo Experimento, y explica en voz alta qué cambió. Si
puedes justificar el resultado sin releer este notebook, ya
dominas el núcleo de la masterclass.

## 🎯 Autoevaluación

Respóndelas sin mirar atrás. No necesitas frases perfectas: explica el mecanismo con tus palabras.

**1. ¿Por qué se separa un conjunto de validation además de train y test?**

A. Porque el modelo necesita más datos para aprender
B. Para decidir arquitectura, dropout, paciencia o umbral sin tocar test
C. Porque test siempre debe ser más grande que train
D. Validation y test son el mismo conjunto con otro nombre

<details>
<summary><b>Ver respuesta</b></summary>

**B.** Validation guía decisiones humanas de configuración durante el desarrollo; test se abre una sola vez, al final, para no contaminar la medición.

</details>

**2. ¿Con qué datos debe ajustarse (`fit`) el `StandardScaler`?**

A. Con todo el dataset, antes de dividir
B. Solo con train
C. Solo con test
D. Con train y validation juntos, pero nunca con test

<details>
<summary><b>Ver respuesta</b></summary>

**B.** Ajustar el scaler con datos fuera de train filtra información del examen final al preprocesado, aunque el modelo nunca 'vea' esas etiquetas directamente.

</details>

**3. ¿Qué hace `restore_best_weights=True` en `EarlyStopping`?**

A. Reinicia los pesos a valores aleatorios al terminar
B. Guarda los pesos de la última época, sea buena o mala
C. Recupera los pesos de la época con mejor `val_loss` observada
D. Congela los pesos de la primera época como referencia

<details>
<summary><b>Ver respuesta</b></summary>

**C.** Sin esta opción, el modelo se quedaría con los pesos de la última época entrenada, que puede ser peor que una anterior si ya venía empeorando.

</details>

**4. En el experimento A/B, la variante A (128→64, sin Dropout, sin EarlyStopping) muestra `loss` de train bajando mucho mientras `val_loss` sube tras cierta época. ¿Qué está pasando?**

A. Infraajuste
B. Sobreajuste: la red memoriza train y deja de generalizar
C. Un error en el código, esa combinación no debería ocurrir
D. El learning rate es demasiado bajo

<details>
<summary><b>Ver respuesta</b></summary>

**B.** Es la firma clásica del sobreajuste: la red sigue reduciendo el error en los datos que ve, pero empeora en datos nuevos porque memorizó detalles de train.

</details>

**5. ¿Qué optimizer y qué loss se usaron para compilar la red en este notebook, y por qué (según lo visto en Notebooks 2 y 3)?**

A. SGD puro y MSE, porque son los más simples
B. Adam y binary cross-entropy, porque Adam adapta el learning rate por parámetro y BCE penaliza probabilidades mal calibradas en clasificación binaria
C. Momentum y categorical cross-entropy, porque hay más de dos clases
D. Adam y accuracy, porque accuracy es la métrica que de verdad se minimiza

<details>
<summary><b>Ver respuesta</b></summary>

**B.** Adam (NB3) combina momentum y tasas de aprendizaje adaptativas por parámetro; binary cross-entropy (NB2) es la loss correcta para clasificación binaria con salida Sigmoid. Accuracy es una métrica de observación, no la función que se minimiza.

</details>

**6. Un modelo alcanza 99% de accuracy en train y 71% en validation. ¿Qué está pasando y qué probarías primero?**

<details>
<summary><b>Qué debería incluir una buena respuesta</b></summary>

- Nombra el fenómeno: sobreajuste (la red memoriza train, no generaliza).
- Propone reducir capacidad (menos neuronas/capas) o subir Dropout.
- Propone activar o endurecer EarlyStopping (menor patience) para no seguir entrenando tras el punto de mejor val_loss.
- Menciona que también podría deberse a muy pocos datos de train para la complejidad del modelo.

</details>

**7. Explica a alguien no técnico por qué comparamos la red neuronal contra una regresión logística simple en vez de confiar directamente en la red.**

<details>
<summary><b>Qué debería incluir una buena respuesta</b></summary>

- Deja claro que un modelo complejo no es automáticamente mejor; hay que demostrarlo con datos.
- Menciona que el baseline es más barato, más rápido de entrenar y más fácil de explicar a terceros.
- Explica que si el baseline empata o gana, la conclusión profesional es usar el modelo simple, no forzar la red.

</details>

In [ ]:
celebrate(
    "🎉 ¡Enhorabuena! Completaste los 4 notebooks de NeuroTrain Lab 🎉",
    "Del perceptrón al entrenamiento con control de sobreajuste: ya conoces "
    "todo el camino. Ahora abre la app Streamlit y pon a prueba tus variantes "
    "A y B en el Modo Experimento.",
)